#Query orders data as JSON strings

In [0]:
select * from gizmobox.bronze.v_orders

In [0]:
select value:order_id AS order_id from gizmobox.bronze.v_orders

In [0]:
select 
value:items[0] AS item_1 ,
value:items[1] AS item_2 
from gizmobox.bronze.v_orders

In [0]:
select 
value:items[0].item_id::Integer AS item_1 
-- value:items[1] AS item_2 
from gizmobox.bronze.v_orders

##Transform order data
missing colon for order date - few

In [0]:
create or replace temporary view v_orders_fixed as
select 
value,
regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date":"\$1"') AS fixed_value
from gizmobox.bronze.v_orders

In [0]:
select schema_of_json(fixed_value) as schema,fixed_value from v_orders_fixed

In [0]:
select from_json(fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as schema,fixed_value from v_orders_fixed

In [0]:
create table gizmobox.silver.orders_json as
select from_json(fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as JSON_value from v_orders_fixed

In [0]:
select * from gizmobox.silver.orders_json